In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%sql
CREATE WIDGET TEXT storageName default="adlsentregafinal";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

##External Location

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `cred-adls-entregafinal`)
COMMENT 'Ubicación externa para las tablas delta del metastore';

CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `cred-adls-entregafinal`)
COMMENT 'Ubicación externa para archivos raw del Data Lake';

CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `cred-adls-entregafinal`)
COMMENT 'Ubicación externa para tablas bronze del Data Lake';

CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `cred-adls-entregafinal`)
COMMENT 'Ubicación externa para tablas silver del Data Lake';

CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `cred-adls-entregafinal`)
COMMENT 'Ubicación externa para tablas gold del Data Lake';

##Catálogo

In [0]:
%sql
DROP CATALOG IF EXISTS retail_dev CASCADE;

CREATE CATALOG IF NOT EXISTS retail_dev
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/carpeta_para_catalogo'
COMMENT 'Catálogo para la arquitectura medallion del ambiente de dev';

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/", True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/", True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/", True)

##Esquemas

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_dev.raw;
CREATE SCHEMA IF NOT EXISTS retail_dev.bronze;
CREATE SCHEMA IF NOT EXISTS retail_dev.silver;
CREATE SCHEMA IF NOT EXISTS retail_dev.golden;

##Tablas Bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail_dev.bronze.clientes (
cliente_id STRING,
nombre STRING,
email STRING,
telefono STRING,
estado STRING,
fecha_registro STRING,
edad STRING,
genero STRING,
codigo_postal STRING,
ingestion_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/clientes";

CREATE TABLE IF NOT EXISTS retail_dev.bronze.productos (
producto_id STRING,
sku STRING,
producto STRING,
categoria STRING,
marca STRING,
costo STRING,
precio STRING,
activo STRING,
fecha_alta STRING,
ingestion_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/productos";

CREATE TABLE IF NOT EXISTS retail_dev.bronze.ordenes (
orden_id STRING,
cliente_id STRING,
fecha_orden STRING,
fecha_entrega STRING,
estatus STRING,
metodo_pago STRING,
canal STRING,
estado_envio STRING,
costo_envio STRING,
ingestion_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/ordenes";

CREATE TABLE IF NOT EXISTS retail_dev.bronze.detalle_ordenes (
detalle_id STRING,
orden_id STRING,
producto_id STRING,
cantidad STRING,
precio_unitario STRING,
descuento STRING,
ingestion_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/detalle_ordenes";

CREATE TABLE IF NOT EXISTS retail_dev.bronze.inventario (
inventario_id STRING,
producto_id STRING,
almacen STRING,
stock_actual STRING,
stock_minimo STRING,
fecha_actualizacion STRING,
ingestion_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/inventario";

##Tablas Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail_dev.silver.ventas_detalle_limpias (
orden_id INTEGER,
cliente_id INTEGER,
nombre_cliente STRING,
email STRING,
estado_cliente STRING,
fecha_orden DATE,
fecha_entrega DATE,
estatus STRING,
metodo_pago STRING,
canal STRING,
estado_envio STRING,
producto_id INTEGER,
sku STRING,
producto STRING,
categoria STRING,
marca STRING,
cantidad INTEGER,
precio_unitario DOUBLE,
descuento DOUBLE,
importe_bruto DOUBLE,
importe_descuento DOUBLE,
importe_neto DOUBLE,
dias_entrega INTEGER,
orden_retrasada STRING
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/ventas_detalle_limpias";

CREATE TABLE IF NOT EXISTS retail_dev.silver.inventario_limpio (
inventario_id INTEGER,
producto_id INTEGER,
sku STRING,
producto STRING,
categoria STRING,
marca STRING,
almacen STRING,
stock_actual INTEGER,
stock_minimo INTEGER,
fecha_actualizacion DATE,
estatus_inventario STRING,
valor_inventario DOUBLE
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/inventario_limpio";

##Tabla Golden

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail_dev.golden.resumen_retail (
categoria STRING,
estado_envio STRING,
total_ordenes BIGINT,
clientes_unicos BIGINT,
unidades_vendidas BIGINT,
venta_total DOUBLE,
ticket_promedio DOUBLE,
productos_bajo_stock BIGINT,
ordenes_retrasadas BIGINT
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/resumen_retail";

##Validación

In [0]:
%sql
SHOW TABLES IN retail_dev.bronze;
SHOW TABLES IN retail_dev.silver;
SHOW TABLES IN retail_dev.golden;